# 27 — MCP Sampling and Roots

Every primitive so far flows client → server: the client lists/calls
tools, reads resources, fetches prompts. **Sampling** and **roots** flip
that direction -- the *server* asks the *client* for something. Both
require a human in the loop per the spec's security guidance, and both are
**client** capabilities (declared by whoever embeds the MCP client, not by
the server).

## Sampling: servers borrowing the client's LLM

Sampling lets a tool implementation request an LLM completion *through the
client* instead of calling a model API directly -- no server-side API key
needed, and the client/user stays in control of which model actually runs
and whether the request is approved at all.

```json
// Server -> Client
{
  "jsonrpc": "2.0", "id": 1, "method": "sampling/createMessage",
  "params": {
    "messages": [{ "role": "user", "content": { "type": "text", "text": "What is the capital of France?" } }],
    "modelPreferences": { "hints": [{ "name": "claude-3-sonnet" }], "intelligencePriority": 0.8 },
    "maxTokens": 100
  }
}
```

`modelPreferences` is an abstraction, not a model name: `costPriority`,
`speedPriority`, and `intelligencePriority` (0-1 each) plus optional
`hints` the client may map to whatever equivalent model it actually has
access to. Per spec, the client **SHOULD** show the user the request (and
the generated response) for review before anything is sent back to the
server.